# CRZ2025 Shock Data

## Getting R007 (National Interbank Funding Center)

In [68]:
import os
import akshare as ak
import pandas as pd
import time
import numpy as np
from functools import reduce


os.chdir('/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data')
print("Current Directory:", os.getcwd())


Current Directory: /Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data


In [69]:
# 1. Create a list of years to loop through
years = range(2000, 2026) 
all_data = []

print("Starting download loop...")

for year in years:
    start_date = f"{year}0101"
    end_date = f"{year}1231"
    
    print(f"Fetching data for {year}...")
    
    try:
        # Fetch data for the specific year
        df = ak.repo_rate_hist(start_date=start_date, end_date=end_date)
        
        # Optional: Add a 'Year' column if you want to track it explicitly
        # df['Year'] = year 
        
        all_data.append(df)
        
        # Pause briefly to be polite to the server (prevents blocking)
        time.sleep(1) 
        
    except Exception as e:
        print(f"Error fetching data for {year}: {e}")
        # Continue to the next year even if one fails

# 2. Combine all years into one DataFrame
if all_data:
    full_repo_df = pd.concat(all_data, ignore_index=True)
    
    # 3. Clean duplicates (just in case) and Sort
    # The date column name might vary slightly; usually it is 'date' or '日期'
    # Check the column names if sort fails.
    if '日期' in full_repo_df.columns:
        full_repo_df.rename(columns={'日期': 'date'}, inplace=True)
        
    full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])
    full_repo_df.sort_values('date', inplace=True)
    
    # 4. Filter for R007 specifically
    # The column for the rate might be '收盘' (Close) or similar.
    # Often the dataframe contains multiple repo types. 
    # If the output contains a "symbol" or "code" column, filter for "R-007" or similar.
    # (Inspect full_repo_df.head() to be sure of the column name for 7-day repo)

    print("Download Complete!")
    print(full_repo_df.head())
    print(f"Total rows: {len(full_repo_df)}")
    

else:
    print("No data fetched.")


Starting download loop...
Fetching data for 2000...
Fetching data for 2001...
Fetching data for 2002...
Fetching data for 2003...
Fetching data for 2004...
Fetching data for 2005...
Fetching data for 2006...
Fetching data for 2007...
Fetching data for 2008...
Fetching data for 2009...
Fetching data for 2010...
Fetching data for 2011...
Fetching data for 2012...
Fetching data for 2013...
Fetching data for 2014...
Fetching data for 2015...
Fetching data for 2016...
Fetching data for 2017...
Fetching data for 2018...
Fetching data for 2019...
Fetching data for 2020...
Fetching data for 2021...
Fetching data for 2022...
Fetching data for 2023...
Fetching data for 2024...
Fetching data for 2025...
Download Complete!
        date  FR001  FR007  FR014  FDR001  FDR007  FDR014
0 2000-01-04    NaN   2.57    NaN     NaN     NaN     NaN
1 2000-01-05    NaN   2.56    NaN     NaN     NaN     NaN
2 2000-01-06    NaN   2.57    NaN     NaN     NaN     NaN
3 2000-01-07    NaN   2.58    NaN     NaN     N

In [70]:
full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])


In [71]:
full_repo_df['quarter'] = full_repo_df['date'].dt.to_period('Q')
china_r007_df = full_repo_df.groupby('quarter')['FR007'].mean().reset_index()
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)

print(china_r007_df.head())

  quarter     FR007
0  2000Q1  2.506311
1  2000Q2  2.398145
2  2000Q3  2.332576
3  2000Q4  2.360902
4  2001Q1  2.501951


## Getting CPI (Jin10)

In [72]:
cpi_df = ak.macro_china_cpi_yearly()


In [73]:
# Remove unnecessary columns and rename to English
cpi_df = cpi_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "cpi"})


In [74]:
# Convert date into proper format
cpi_df['date'] = pd.to_datetime(cpi_df['date'])

In [75]:
# Remove values before 2000
cpi_df = cpi_df[cpi_df['date'].dt.year >= 2000]

In [76]:
# Convert into quarter format
cpi_df['quarter'] = cpi_df['date'].dt.to_period('Q')

In [77]:
cpi_df = cpi_df.groupby('quarter')['cpi'].mean().reset_index()

## Cleaning Real GDP (National Bureau of Statistics)

In [78]:
gdp_df = pd.read_excel("China_RealGDP.xlsx")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [79]:
gdp_df = gdp_df[["指标名称", "中国:GDP:不变价:当季同比"]].rename(columns={"指标名称": "date", "中国:GDP:不变价:当季同比": "realgdpgrowth"})

In [80]:
# Remove the notes rows
gdp_df = gdp_df.iloc[4:-2]

In [81]:
gdp_df['date'] = pd.to_datetime(gdp_df['date']).dt.to_period('Q')

In [82]:
gdp_df = gdp_df[gdp_df['date'].dt.year >= 2000]
gdp_df['realgdpgrowth'] = pd.to_numeric(gdp_df['realgdpgrowth'], errors='coerce')

In [83]:
gdp_df = gdp_df.rename(columns={'date': 'quarter'})

## Combining all three data

In [84]:
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)
cpi_df['quarter'] = cpi_df['quarter'].astype(str)
gdp_df['quarter'] = gdp_df['quarter'].astype(str)

In [85]:
# 1. Put dataframes in a list
df_crz2025 = [china_r007_df, cpi_df, gdp_df]

# 2. Define a merging function (Inner Join = Intersection of dates)
# This keeps only rows where 'quarter' exists in ALL dataframes
df_crz2025 = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='inner'), df_crz2025)

# 3. specific cleanup
# Set quarter as the index and sort it chronologically
df_crz2025 = df_crz2025.set_index('quarter').sort_index()


In [86]:
df_crz2025 = df_crz2025.reset_index()

In [87]:
df_crz2025['quarter'] = df_crz2025['quarter'].astype(str)

## Adding Inflation and RGDP Target

In [88]:

# 1. Define the targets (Midpoints used for ranges)
# Source: Government Work Reports
china_gdp_targets = {
    2000: 7.0, 2001: 7.0, 2002: 7.0, 2003: 7.0, 2004: 7.0,
    2005: 8.0, 2006: 8.0, 2007: 8.0, 2008: 8.0, 2009: 8.0,
    2010: 8.0, 2011: 8.0, 2012: 7.5, 2013: 7.5, 2014: 7.5,
    2015: 7.0, 
    2016: 6.75, # Range 6.5-7.0 -> Midpoint 6.75
    2017: 6.5, 
    2018: 6.5, 
    2019: 6.25, # Range 6.0-6.5 -> Midpoint 6.25
    2020: 6.0, # No official target set Set to 6.0 for calculation
    2021: 6.0,  # "Above 6.0" -> Treated as 6.0 baseline
    2022: 5.5, 
    2023: 5.0,
    2024: 5.0,
    2025: 5.0
}

china_cpi_targets = {
    2000: 2.0, 2001: 1.5, 2002: 1.5, 2003: 1.0, 2004: 3.0,
    2005: 4.0, 2006: 3.0, 2007: 3.0, 2008: 4.8, 2009: 4.0,
    2010: 3.0, 2011: 4.0, 2012: 4.0, 2013: 3.5, 2014: 3.5,
    2015: 3.0, 2016: 3.0, 2017: 3.0, 2018: 3.0, 2019: 3.0,
    2020: 3.5, 2021: 3.0, 2022: 3.0, 2023: 3.0, 2024: 3.0,
    2025: 2.0
}

years = df_crz2025['quarter'].str[:4].astype(int)
# 2. Extract the year from your PeriodIndex (assuming index is '2000Q1' etc.)
# If your index is named something else, change 'df_final.index'
df_crz2025['target_gdp'] = years.map(china_gdp_targets)
df_crz2025['target_cpi'] = years.map(china_cpi_targets)

# 4. Calculate the Gap immediately
df_crz2025['gdp_gap'] = df_crz2025['realgdpgrowth'] - df_crz2025['target_gdp']
df_crz2025['cpi_gap'] = df_crz2025['cpi'] - df_crz2025['target_cpi']


In [89]:
df_crz2025.to_csv('crz2025.csv', index=False)

## (Value-Added) Industrial Production

In [90]:
ip_df = ak.macro_china_industrial_production_yoy()

In [91]:
# Remove unnecessary columns and rename to English
ip_df = ip_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "ip"})


In [92]:
# Convert date into proper format
ip_df['date'] = pd.to_datetime(ip_df['date'])

In [93]:
# Remove values before 2000
ip_df = ip_df[ip_df['date'].dt.year >= 2000]

In [94]:
# Convert into quarter format
ip_df['quarter'] = ip_df['date'].dt.to_period('Q')

In [95]:
# Average within each quarter
ip_df = ip_df.groupby('quarter')['ip'].mean().reset_index()

## Composite Monetary Policy Index (CMPI)

In [96]:
import re

# Get all sheet names from CMPI.xlsx
excel_file = pd.ExcelFile("CMPI.xlsx")
sheet_names = excel_file.sheet_names

# Dictionary to store all dataframes
cmpi_dfs = {}

# Loop through each sheet
for sheet_name in sheet_names:
    # 1) Read sheet
    df = pd.read_excel("CMPI.xlsx", sheet_name=sheet_name)

    # 2) Remove first row
    df = df.iloc[1:].copy()

    # 3) Rename first column to date + parse datetime
    first_col = df.columns[0]
    df.rename(columns={first_col: "date"}, inplace=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # 4) Keep year >= 2000
    df = df[df["date"].dt.year >= 2000]

    # 5) Quarter
    df["quarter"] = df["date"].dt.to_period("Q")
    # Convert all non-date/quarter columns to numeric
    candidate_cols = [c for c in df.columns if c not in ["date", "quarter"]]
    for c in candidate_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 6) Quarterly mean for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        df = df.groupby("quarter")[numeric_cols].mean().reset_index()
    else:
        df = df[["quarter"]].drop_duplicates().reset_index(drop=True)

    # 7) Clean df variable name: remove Chinese/ASCII parentheses
    clean_sheet_name = re.sub(r"[（）()]", "", sheet_name).strip()
    df_name = f"{clean_sheet_name}_df"

    # 8) Save to dict + globals
    cmpi_dfs[df_name] = df
    globals()[df_name] = df

print(f"Processed {len(sheet_names)} sheets from CMPI.xlsx")
print("Available dataframes:", list(cmpi_dfs.keys()))

Processed 13 sheets from CMPI.xlsx
Available dataframes: ['正回购利率_df', '逆回购利率_df', '国库现金利率_df', '常备借贷便利利率_df', '短期流动性工具利率_df', '央行票据发行利率_df', '定期存款基准利率_df', '金融机构在央行存款利率_df', '中长期贷款基准利率_df', '中期借贷便利MLF利率_df', '抵押补充贷款利率_df', '贷款市场报价利率LPR_df', '存款准备金率RRR_df']


In [97]:
正回购利率_df.rename(columns={'Unnamed: 1': 'Repo14days', 'Unnamed: 2': 'Repo21days', 'Unnamed: 3': 'Repo28days', 'Unnamed: 4': 'Repo91days', 'Unnamed: 5': 'Repo182days', 'Unnamed: 6': 'Repo364days'}, inplace=True)
逆回购利率_df.rename(columns={'Unnamed: 1': 'ReRepo7days', 'Unnamed: 2': 'ReRepo14days', 'Unnamed: 3': 'ReRepo21days', 'Unnamed: 4': 'ReRepo28days', 'Unnamed: 5': 'ReRepo91days'}, inplace=True)
国库现金利率_df.rename(columns={'Unnamed: 1': 'TC3months', 'Unnamed: 2': 'TC6months', 'Unnamed: 3': 'TC9months'}, inplace=True)
常备借贷便利利率_df.rename(columns={'Unnamed: 1': 'SLFovernight', 'Unnamed: 2': 'SLF7days', 'Unnamed: 3': 'SLF1month'}, inplace=True)
短期流动性工具利率_df.rename(columns={'表五：短期流动性调节工具（SLO）操作利率': 'SLO', 'Unnamed: 2': 'ReverseSLO'}, inplace=True) 
央行票据发行利率_df.rename(columns={'Unnamed: 1': 'CBB3months', 'Unnamed: 2': 'CBB6months', 'Unnamed: 3': 'CBB1year', 'Unnamed: 4': 'CBB3years'}, inplace=True)
定期存款基准利率_df.rename(columns={'Unnamed: 1': 'TD3months', 'Unnamed: 2': 'TD1year'}, inplace=True)
金融机构在央行存款利率_df.rename(columns={'Unnamed: 1': 'RequiredReserve', 'Unnamed: 2': 'ExcessReserve'}, inplace=True)
中长期贷款基准利率_df.rename(columns={'Unnamed: 1': 'ML1-3years', 'Unnamed: 2': 'ML3-5years'}, inplace=True)
中期借贷便利MLF利率_df.rename(columns={'Unnamed: 1': 'MLF3months', 'Unnamed: 2': 'MLF6months', 'Unnamed: 3': 'MLF1year'}, inplace=True)
抵押补充贷款利率_df.rename(columns={'Unnamed: 1': 'PSL'}, inplace=True)
贷款市场报价利率LPR_df.rename(columns={'Unnamed: 1': 'LPR1year', 'Unnamed: 2': 'LPR5years'}, inplace=True)
存款准备金率RRR_df.rename(columns={'Unnamed: 1': 'RRRLarge', 'Unnamed: 2': 'RRRSmall'}, inplace=True)

In [98]:
# Join all CMPI dataframes by quarter
cmpi_list = list(cmpi_dfs.values())
cmpi_df = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='outer'), cmpi_list)

# Get all numeric columns (excluding quarter)
numeric_cols = cmpi_df.select_dtypes(include=[np.number]).columns.tolist()

# Standardize each numeric variable: (x - mean) / std
for col in numeric_cols:
    mean_val = cmpi_df[col].mean()
    std_val = cmpi_df[col].std()
    cmpi_df[f'{col}_standardized'] = (cmpi_df[col] - mean_val) / std_val

# Get all standardized columns
standardized_cols = [col for col in cmpi_df.columns if 'standardized' in col]

# Calculate CMPI as the average of all standardized variables for each quarter
cmpi_df['cmpi'] = cmpi_df[standardized_cols].mean(axis=1)

# Convert quarter to string for easier merging later
cmpi_df['quarter'] = cmpi_df['quarter'].astype(str)

print("CMPI dataframe created successfully with standardized values!")
print(f"Shape: {cmpi_df.shape}")
print(cmpi_df[['quarter', 'cmpi']].head())

CMPI dataframe created successfully with standardized values!
Shape: (105, 76)
  quarter      cmpi
0  2000Q1  0.400283
1  2000Q2  0.343119
2  2000Q3  0.210993
3  2000Q4  0.177851
4  2001Q1  0.350013


## CNYUSD Spot Rate

In [99]:
spot_df = pd.read_excel("CNYUSD_Spot_Rate.xlsx")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [100]:
spot_df = spot_df[["指标名称", "CFETS:即期汇率:美元兑人民币"]].rename(columns={"指标名称": "date", "CFETS:即期汇率:美元兑人民币": "CNYUSDSpot"})

In [101]:
# Remove the notes rows
spot_df = spot_df.iloc[4:-2]

In [102]:
spot_df['date'] = pd.to_datetime(spot_df['date']).dt.to_period('Q')

In [103]:
spot_df = spot_df[spot_df['date'].dt.year >= 2000]
spot_df['CNYUSDSpot'] = pd.to_numeric(spot_df['CNYUSDSpot'], errors='coerce')

In [104]:
spot_df = spot_df.rename(columns={'date': 'quarter'})

In [105]:
spot_df = spot_df.groupby('quarter')['CNYUSDSpot'].mean().reset_index()

## CNYUSDCPR

In [106]:
cpr_df = ak.currency_boc_sina(symbol="美元", start_date="20000101", end_date="20251231")

In [107]:
cpr_df = cpr_df[["日期", "央行中间价"]].rename(columns={"日期": "date", "央行中间价": "CNYUSDCPR"})

In [108]:
cpr_df['date'] = pd.to_datetime(cpr_df['date']).dt.to_period('Q')

In [109]:
cpr_df = cpr_df[cpr_df['date'].dt.year >= 2000]
cpr_df['CNYUSDCPR'] = pd.to_numeric(cpr_df['CNYUSDCPR'], errors='coerce')

In [110]:
cpr_df = cpr_df.rename(columns={'date': 'quarter'})

In [111]:
cpr_df = cpr_df.groupby('quarter')['CNYUSDCPR'].mean().reset_index()

In [112]:
cpr_df['CNYUSDCPR'] = cpr_df['CNYUSDCPR'] / 100

## PPI

In [113]:
ppi_df = ak.macro_china_ppi_yearly()

In [114]:
# Remove unnecessary columns and rename to English
ppi_df = ppi_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "ppi"})

In [115]:
# Convert date into proper format
ppi_df['date'] = pd.to_datetime(ppi_df['date'])

In [116]:
# Remove values before 2000
ppi_df = ppi_df[ppi_df['date'].dt.year >= 2000]

In [117]:
# Convert into quarter format
ppi_df['quarter'] = ppi_df['date'].dt.to_period('Q')

In [118]:
# Average within each quarter
ppi_df = ppi_df.groupby('quarter')['ppi'].mean().reset_index()

## Combine Final (Temp)

In [119]:
# Convert all quarter columns to string for consistent merging
ip_df['quarter'] = ip_df['quarter'].astype(str)
cmpi_df['quarter'] = cmpi_df['quarter'].astype(str)
spot_df['quarter'] = spot_df['quarter'].astype(str)
cpr_df['quarter'] = cpr_df['quarter'].astype(str)
ppi_df['quarter'] = ppi_df['quarter'].astype(str)

# Select only quarter and cmpi from cmpi_df
cmpi_subset = cmpi_df[['quarter', 'cmpi']]

# Merge all dataframes into df_crz2025
df_list = [df_crz2025, ip_df, cmpi_subset, spot_df, cpr_df, ppi_df]
df_final = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='outer'), df_list)

# Sort by quarter
df_final = df_final.sort_values('quarter').reset_index(drop=True)

print("Updated df_final shape:", df_final.shape)
print(df_final.head())
df_final.to_csv('df_final.csv', index=False)

Updated df_final shape: (105, 13)
  quarter     FR007       cpi  realgdpgrowth  target_gdp  target_cpi  gdp_gap  \
0  2000Q1  2.506311 -0.166667            8.8         7.0         2.0      1.8   
1  2000Q2  2.398145 -0.133333            9.2         7.0         2.0      2.2   
2  2000Q3  2.332576  0.433333            8.9         7.0         2.0      1.9   
3  2000Q4  2.360902  0.433333            7.6         7.0         2.0      0.6   
4  2001Q1  2.501951  0.900000            9.5         7.0         1.5      2.5   

    cpi_gap         ip      cmpi  CNYUSDSpot  CNYUSDCPR       ppi  
0 -2.166667   9.433333  0.400283    8.278607   8.278586  0.066667  
1 -2.133333  11.600000  0.343119    8.277993   8.278035  2.333333  
2 -1.566667  12.600000  0.210993    8.279175   8.279151  3.800000  
3 -1.566667  11.333333  0.177851    8.277573   8.277590  3.600000  
4 -0.600000  10.566667  0.350013    8.277286   8.277312  1.700000  
